# GP08 — Buffer Analysis

Η ροή είναι **Πρόβλεψε → Εκτέλεσε → Έλεγξε → Ερμήνευσε**. Οι ζώνες μοντελοποιούν ευθύγραμμη εγγύτητα και όχι χρόνο μετακίνησης ή οδική προσβασιμότητα.


## 1. Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv, json

candidates = (Path("downloads/python/data"), Path("data"), Path("../../data"))
DATA_ROOT = next((p for p in candidates if (p / "gp08").is_dir()), Path("data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_URLS = {
    "roads.geojson": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp08/roads.geojson",
    "services.csv": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp08/services.csv",
    "study_area.geojson": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp08/study_area.geojson"
}

def valid_cached(path):
    if not path.exists() or path.stat().st_size == 0: return False
    try:
        if path.suffix.lower() == ".csv":
            with path.open(encoding="utf-8-sig", newline="") as f:
                reader = csv.DictReader(f)
                required = {"service_id", "name", "service_type", "longitude", "latitude"}
                return required.issubset(reader.fieldnames or []) and next(reader, None) is not None
        if path.suffix.lower() == ".geojson":
            return json.loads(path.read_text(encoding="utf-8")).get("type") == "FeatureCollection"
        return True
    except Exception: return False

for filename, url in DATA_URLS.items():
    target = DATA_ROOT / "gp08" / filename
    if valid_cached(target):
        print(f"[OK] Ήδη διαθέσιμο: {target}"); continue
    target.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url, timeout=30) as response: payload = response.read()
        if not payload: raise RuntimeError("κενή απάντηση")
        target.write_bytes(payload)
        if not valid_cached(target): raise RuntimeError("μη έγκυρο αρχείο")
        print(f"[OK] Κατέβηκε: {target}")
    except Exception as exc:
        if target.exists(): target.unlink()
        raise RuntimeError(f"Αποτυχία προετοιμασίας δεδομένων: {url} — {exc}") from exc


## 2. Ερώτημα και έλεγχος inputs

Πριν γράψεις κώδικα, γράψε τι geometry και μονάδα περιμένεις για buffer 500 m γύρω από σχολείο.


In [ ]:
import pandas as pd
import geopandas as gpd
root = DATA_ROOT / "gp08"
services_df = pd.read_csv(root / "services.csv")
services = gpd.GeoDataFrame(services_df, geometry=gpd.points_from_xy(services_df.longitude, services_df.latitude), crs="EPSG:4326")
roads = gpd.read_file(root / "roads.geojson")
area = gpd.read_file(root / "study_area.geojson")
assert services.crs.to_epsg() == 4326
assert services.geometry.geom_type.eq("Point").all()
assert roads.geometry.geom_type.eq("LineString").all()
assert area.geometry.geom_type.eq("Polygon").all()
assert services.geometry.notna().all() and services.geometry.is_valid.all()
print("inputs:", services.shape, roads.shape, area.shape)


## 3. Καθοδηγούμενο παράδειγμα: σχολικά buffers 500 m

Το `500` είναι μέτρα μόνο μετά τον μετασχηματισμό στο EPSG:2100. Κράτησε τα αρχικά points και δούλεψε σε αντίγραφο.


In [ ]:
target_crs = "EPSG:2100"
services_m = services.to_crs(target_crs)
roads_m = roads.to_crs(target_crs)
area_m = area.to_crs(target_crs)
school_buffers = services_m.loc[services_m["service_type"].eq("school")].copy()
school_buffers["school_id"] = school_buffers["service_id"]
school_buffers["buffer_m"] = 500
school_buffers["geometry"] = school_buffers.geometry.buffer(500)
assert services_m.crs.to_epsg() == 2100
assert school_buffers.geometry.geom_type.isin(["Polygon", "MultiPolygon"]).all()
assert school_buffers.geometry.notna().all() and school_buffers.geometry.is_valid.all()
assert (school_buffers.geometry.area > 0).all()
print("school buffers:", len(school_buffers))
print("areas:", school_buffers.geometry.area.tolist())


## 4. Ερμηνεία και οπτικός έλεγχος

Έλεγξε rows, geometry type, validity, emptiness, CRS, bounds και area. Η εικόνα βοηθά να ελέγξεις θέση και μέγεθος, αλλά δεν αποδεικνύει μόνη της πλήρη χωρική ορθότητα.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 5))
area_m.boundary.plot(ax=ax, color="0.5", linewidth=1, label="study area")
school_buffers.plot(ax=ax, alpha=0.35, edgecolor="tab:blue", column="school_id", legend=True)
services_m.plot(ax=ax, color="black", markersize=18)
ax.set_title("Σχολεία και buffers 500 m")
ax.set_aspect("equal")
plt.show()


## 5. Ελεγχόμενη σύγκριση ακτίνας

Πρόβλεψε πρώτα. Στο επόμενο κελί συμπλήρωσε την ακτίνα, τη σύγκριση εμβαδών και έναν έλεγχο ότι το αποτέλεσμα έχει το αναμενόμενο πλήθος και geometry. Με τα πραγματικά δεδομένα οι ζώνες 250 m και 500 m δεν επικαλύπτονται και το πλήθος matched υπηρεσιών δεν αλλάζει· αυτό είναι σωστό αποτέλεσμα, όχι αποτυχία.


In [ ]:
# TODO: συμπλήρωσε radius_250, έναν ουσιαστικό υπολογισμό και τους ελέγχους.
radius_250 = None
# Υπόδειξη: σύγκρινε τα εμβαδά των δύο GeoSeries και κατέληξε σε έναν έλεγχο με .all().
area_comparison = None
# Υπόδειξη: έλεγξε ότι οι νέες γεωμετρίες έχουν τον αναμενόμενο τύπο και είναι έγκυρες.
geometry_check = None
if radius_250 is None or area_comparison is None or geometry_check is None:
    raise NotImplementedError("Συμπλήρωσε την ακτίνα, τη σύγκριση εμβαδών και τον έλεγχο geometry")
school_buffers_250 = school_buffers.copy()
school_buffers_250["buffer_m"] = radius_250
school_buffers_250["geometry"] = services_m.loc[school_buffers.index].geometry.buffer(radius_250)
print("areas:", school_buffers_250.geometry.area.tolist(), school_buffers.geometry.area.tolist())
assert area_comparison
assert geometry_check


## 6. Δρόμοι, πολλαπλές αντιστοιχίσεις και dissolve

Το παρακάτω καθοδηγούμενο παράδειγμα χρησιμοποιεί τους δύο δρόμους με ακτίνα 500 m για να συγκρίνεις match rows, μοναδικές υπηρεσίες και υπηρεσίες που αντιστοιχούν σε διαφορετικούς road buffers. Μετά εξήγησε τι αλλάζει με dissolve. Με τα δεδομένα του μαθήματος αναμένονται 5 match rows και 4 μοναδικές υπηρεσίες· η S01 αντιστοιχίζεται σε δύο οδικούς buffers. Χρησιμοποίησε αυτά τα αποτελέσματα ως έλεγχο ερμηνείας, όχι ως αριθμούς προς απομνημόνευση.


In [ ]:
road_buffers = roads_m.copy()
road_buffers["buffer_m"] = 500
road_buffers["geometry"] = road_buffers.geometry.buffer(500)
road_matches = gpd.sjoin(services_m, road_buffers[["road_id", "geometry"]], how="inner", predicate="within", lsuffix="service", rsuffix="road")
print("road match rows:", len(road_matches))
print("unique services:", road_matches["service_id"].nunique())
print(road_matches.groupby("service_id")["road_id"].nunique())
combined_roads = road_buffers.dissolve()
print("combined road coverage area:", combined_roads.geometry.area.sum())


## 7. Χωρική επιλογή υπηρεσιών

Χρησιμοποίησε `school_id` ως αναγνωριστικό buffer. Το αρχικό `service_id` πρέπει να διατηρείται χωρίς σύγκρουση. Έλεγξε match rows, μοναδικά service IDs, duplicates ανά σχολείο και το `within`.


In [ ]:
matched = gpd.sjoin(services_m, school_buffers[["school_id", "buffer_m", "geometry"]], how="inner", predicate="within", lsuffix="service", rsuffix="school")
print("match rows:", len(matched))
print("unique services:", matched["service_id"].nunique())
print(matched.groupby("service_id")["school_id"].nunique())
assert matched.geometry.geom_type.eq("Point").all()
assert matched.crs.to_epsg() == 2100


## 8. QA, export και round-trip

Έλεγξε CRS, rows, IDs, geometry, `is_valid`, `is_empty`, bounds και θετικό area. Πρόσθεσε οπτικό έλεγχο με το plot. Το export πρέπει να κρατά `buffer_m`, `school_id` και `service_id` όπου υπάρχουν.


In [ ]:
assert school_buffers.crs.to_epsg() == 2100
assert (~school_buffers.geometry.is_empty).all() and school_buffers.geometry.is_valid.all()
assert school_buffers["service_id"].notna().all() and school_buffers["school_id"].notna().all()
assert (school_buffers.geometry.area > 0).all()
export_path = "gp08_school_buffers.gpkg"
school_buffers.to_file(export_path, layer="school_buffers", driver="GPKG")
reloaded = gpd.read_file(export_path, layer="school_buffers")
assert len(reloaded) == len(school_buffers)
assert reloaded.crs.to_epsg() == 2100
assert reloaded.geometry.geom_type.isin(["Polygon", "MultiPolygon"]).all()
assert reloaded.geometry.is_valid.all() and (~reloaded.geometry.is_empty).all()
assert set(reloaded["school_id"]) == set(school_buffers["school_id"])
assert set(reloaded["service_id"]) == set(school_buffers["service_id"])
assert ((reloaded.total_bounds - school_buffers.total_bounds) ** 2 < 1e-6).all()
assert reloaded["buffer_m"].eq(500).all()
print("round-trip passed:", export_path)


## 9. Ανεξάρτητο mini challenge

Διατύπωσε νέο ερώτημα, διαφορετικό από τα σχολικά και οδικά guided examples. Συμπλήρωσε input, απόσταση, χωρική πράξη, prediction, rows, unique IDs, CRS, geometry, validity, bounds, export και περιορισμούς.

Το κελί έχει σκόπιμο `NotImplementedError`: συμπλήρωσέ το πριν εκτελέσεις ολόκληρο το notebook. Δεν είναι απρόσμενο τεχνικό σφάλμα και δεν υπάρχει έτοιμη λύση.


In [ ]:
# TODO: συμπλήρωσε το δικό σου ερώτημα και τα QA checks.
raise NotImplementedError("Συμπλήρωσε το ανεξάρτητο mini challenge πριν συνεχίσεις")


## 10. Τεκμηρίωση αποφάσεων

Πρόσθεσε Markdown cell με source/working CRS, απόσταση και αιτιολόγηση, match rows/unique IDs, geometry checks, export path, πρόβλεψη και παρατήρηση.


## Readiness checkpoint

Μπορείς να εξηγήσεις γιατί το `set_crs` δεν επαναπροβάλλει, γιατί τα match rows μπορεί να διαφέρουν από τα μοναδικά features, τι χάνεται με dissolve και γιατί το buffer δεν αποδεικνύει χρόνο μετακίνησης;
